# Session 19: Pandas practice

In this session we will use Spotify data to practice.

These are the topics to be covered:

- Reading data from JSON files
- Concatenating dataframes
- Filtering dataframes
- Grouping and aggregating data

In [37]:
import pandas as pd

## Exercise 1: Reading data from JSON files

Convert the two JSON files into a single dataframe.

In [38]:
spotify0 = pd.read_json(
    "/Users/dgh/Desktop/pda_mbads/pda1/data/StreamingHistory_music_0.json"
)

spotify1 = pd.read_json(
    "/Users/dgh/Desktop/pda_mbads/pda1/data/StreamingHistory_music_1.json"
)

spotify = pd.concat([spotify0, spotify1])

spotify.head()

,endTime,artistName,trackName,msPlayed
0,2024-06-11 06:27,Nacho Vegas,Lo Que Comen las Brujas,83586
1,2024-06-11 10:21,Nacho Vegas,Lo Que Comen las Brujas,114248
2,2024-06-11 10:23,bigott,She is My Man,14411
3,2024-06-11 10:23,interrogación amor,tú y yo,3482
4,2024-06-11 10:23,Doble Pletina,Cruzo los dedos,115450


## Exercise 2: 

- Who's the artist with the most tracks in the dataset?
- Who's the artist I've listened to for the longest total time?
- What's the song that I've listened to the most times?
- What's the song I've played for the longest total time?

In [ ]:
# artist with most reproductions
spotify["artistName"].value_counts().idxmax()

'Mujeres'

In [ ]:
# artist with highest amount of unique songs in the dataset
spotify.groupby("artistName")["trackName"].nunique().idxmax()

'Los Planetas'

In [ ]:
# artist with longest total reportduciton time

"""
select
    artistName,
    sum(msPlayed) as total
from spotify
group by 1
order by 2 desc
limit 1
"""

spotify.groupby("artistName")["msPlayed"].sum().idxmax()

'Mujeres'

In [ ]:
# song ive listened to the most times

spotify["trackName"].value_counts().idxmax()

# value counts is equivalent to groupby + count

'Unknown Track'

In [ ]:
spotify.groupby("trackName")["msPlayed"].sum().idxmax()

'Un Gesto Brillante'

In [ ]:
# to remove the first row
spotify["trackName"].value_counts()[1:].idxmax()

'Un Gesto Brillante'

## Exercise 3:

How many artists whose songs (individually counted) I have listened to for 1 minute or less?

In [ ]:
# one minute or less

less_1min = spotify[spotify["msPlayed"] <= 60 * 1000]

len(less_1min.groupby("artistName")["trackName"].nunique())

1068

In [ ]:
less_1min["artistName"].nunique()

1068

## Exercise 4: 

What's the artist with the longest song?

In [ ]:
list(spotify["artistName"][spotify["msPlayed"].idxmax()].values)

['Karavana', 'Los Planetas']

## Interlude: datetime objects

In Pandas, we can convert strings to datetime objects using the `pd.to_datetime()` function. This is useful for filtering and manipulating date and time data.

Also, it allows us to extract specific components like year, month, day, hour, etc.

In [48]:
spotify["endTime"].dtype

dtype('O')

In [49]:
spotify["endTime"] = pd.to_datetime(spotify["endTime"])

spotify["endTime"].dtype

dtype('<M8[ns]')

In [50]:
spotify["endTime"] = pd.to_datetime(spotify["endTime"])

spotify["date"] = spotify["endTime"].dt.date
spotify["year"] = spotify["endTime"].dt.year
spotify["month"] = spotify["endTime"].dt.month
spotify["day"] = spotify["endTime"].dt.day
spotify["weekday"] = spotify["endTime"].dt.weekday
spotify["hour"] = spotify["endTime"].dt.hour
spotify["minute"] = spotify["endTime"].dt.minute
spotify["is_weekend"] = spotify["weekday"].isin([5, 6])

spotify.head()

,endTime,artistName,trackName,msPlayed,date,year,month,day,weekday,hour,minute,is_weekend
0,2024-06-11 06:27:00,Nacho Vegas,Lo Que Comen las Brujas,83586,2024-06-11,2024,6,11,1,6,27,False
1,2024-06-11 10:21:00,Nacho Vegas,Lo Que Comen las Brujas,114248,2024-06-11,2024,6,11,1,10,21,False
2,2024-06-11 10:23:00,bigott,She is My Man,14411,2024-06-11,2024,6,11,1,10,23,False
3,2024-06-11 10:23:00,interrogación amor,tú y yo,3482,2024-06-11,2024,6,11,1,10,23,False
4,2024-06-11 10:23:00,Doble Pletina,Cruzo los dedos,115450,2024-06-11,2024,6,11,1,10,23,False


## Exercise 5:

Convert the `weekday` column from 0 to 6 to the actual names of the days of the week (e.g., 0 -> "Monday", 1 -> "Tuesday", etc.).

In [ ]:
df = pd.DataFrame({"a": [None, None, 1]})

df.isna()

,a
0,True
1,True
2,False


In [ ]:
df = df.fillna(0)

,a
0,0.0
1,0.0
2,1.0


In [51]:
spotify["weekday"].unique()

array([1, 2, 3, 4, 0, 5, 6], dtype=int32)

In [52]:
days_dict = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}

spotify["weekday"] = spotify["weekday"].map(days_dict)

spotify["weekday"]

0         Tuesday
1         Tuesday
2         Tuesday
3         Tuesday
4         Tuesday
          ...    
4920    Wednesday
4921    Wednesday
4922    Wednesday
4923    Wednesday
4924    Wednesday
Name: weekday, Length: 14925, dtype: object

## Exercise 6:

What's the total time of music per month I've listened to?

In [ ]:
# in hours
spotify.groupby("month")["msPlayed"].sum() / 60 / 1000 / 60

month
1     30.503988
2     48.942662
3     52.869849
4     52.527817
5     66.953454
6     41.810779
7     52.684615
8     30.414437
9     57.652712
10    58.543177
11    66.913354
12    48.415912
Name: msPlayed, dtype: float64

## Exercise 7:

What's the average number of artists played per day of the week?

In [ ]:
count_artist_per_weekday = spotify.groupby("weekday")["artistName"].value_counts()

df = pd.DataFrame(count_artist_per_weekday).reset_index()

df["weekday"].value_counts()

# create the same but for artists
# and then divide

np.float64(556.1428571428571)

## Exercise 8:

What could you say of my music listening habits based on the hourly data?

In [ ]:
spotify["endTime"].apply(["min", "max"])

min   2024-06-11 06:27:00
max   2025-06-11 16:54:00
Name: endTime, dtype: datetime64[ns]

In [ ]:
hourly_data = spotify.groupby("hour")["msPlayed"].sum() / 60000

hourly_data / 365

hour
0      0.204045
1      0.171704
2      0.165325
3      0.156309
4      0.241119
5      0.289182
6      1.401797
7      3.675117
8      7.356770
9      9.971819
10    11.510899
11     9.468499
12     6.966836
13     9.127363
14     8.753625
15     7.749197
16     7.865077
17     5.287024
18     4.076353
19     1.870038
20     1.016658
21     1.602375
22     0.720944
23     0.335392
Name: msPlayed, dtype: float64

## Exercise 9:

Extract all the distinct artists I have listened to between 0am and 7am

In [ ]:
between_0_7 = spotify[spotify["hour"] <= 7]

# between_0_7 = spotify[spotify['hour'].isin([0,1, 2, 3, 4, 5, 6, 7])]

between_0_7["artistName"].unique()  # count(distinct arstistName)

array(['Nacho Vegas', 'Joaquín Sabina', 'Weezer', 'Alcalá Norte',
       'Sufjan Stevens', 'Van Morrison', 'Wild Honey',
       'Grupo de expertos Solynieve', 'Cariño', 'Family', 'Nick Drake',
       'Andrew Bird', 'Iron & Wine', 'Belle and Sebastian', 'Marcus King',
       'Father John Misty', 'Arthur Verocai', 'Carolina Durante',
       'Drugdealer', 'El Hombre Burbuja', 'Jens Lekman',
       'El Palacio de Linares', 'NATHY PELUSO', 'Yung Bae', 'Lori Meyers',
       'Lin-Manuel Miranda', 'Destroyer', 'Quavo', 'La Bien Querida',
       'Klaus & Kinski', 'Tank and The Bangas', 'Breakbot', 'Kavinsky',
       'Parcels', 'Spiller', 'Mdou Moctar', 'Altin Gün',
       'Fujiya & Miyagi', 'Mujeres', 'Dharmacide', 'JAY-Z', 'Phoenix',
       'El Mató a un Policía Motorizado', 'Triángulo de Amor Bizarro',
       'Manos De Topo', 'LUIS BREA', 'Francisco Nixon', 'Mercromina',
       'Fleet Foxes', 'The Postal Service', 'The Magnetic Fields',
       'The Shins', 'Alvvays', 'Khruangbin', 'The xx', '

## Exercise 10:

What's the month/weekday/hour where I've listened to the most music?

In [ ]:
spotify["endTime"].sample(5)

9120   2025-01-15 10:09:00
3058   2025-05-05 17:51:00
9303   2025-01-22 10:51:00
4148   2025-05-20 17:36:00
787    2025-03-08 11:57:00
Name: endTime, dtype: datetime64[ns]

In [ ]:
# month
print(spotify.groupby("month")["msPlayed"].sum().idxmax())

# weekday
print(spotify.groupby("weekday")["msPlayed"].sum().idxmax())

# hour
print(spotify.groupby("hour")["msPlayed"].sum().idxmax())

5
Wednesday
10


## Exercise 11:

On May 10th 2025, I hosted a party. When did the last guest leave?

In [ ]:
spotify[(spotify["year"] == 2025) & (spotify["month"] == 5) & (spotify["day"] == 11)]

,endTime,artistName,trackName,msPlayed,date,year,month,day,weekday,hour,minute,is_weekend
3631,2025-05-11 00:04:00,NEIL FRANCES,Dumb Love,282728,2025-05-11,2025,5,11,Sunday,0,4,True
3632,2025-05-11 00:10:00,Salif Keita,Madan,359733,2025-05-11,2025,5,11,Sunday,0,10,True
3633,2025-05-11 00:12:00,Jungle,Keep Me Satisfied,145663,2025-05-11,2025,5,11,Sunday,0,12,True
3634,2025-05-11 00:16:00,Joe Crepúsculo,A Fuego,244701,2025-05-11,2025,5,11,Sunday,0,16,True
3635,2025-05-11 00:23:00,New Order,Temptation,419240,2025-05-11,2025,5,11,Sunday,0,23,True
...,...,...,...,...,...,...,...,...,...,...,...,...
3799,2025-05-11 16:01:00,Jens Lekman,Wedding in Finistère,204680,2025-05-11,2025,5,11,Sunday,16,1,True
3800,2025-05-11 16:05:00,Altin Gün,Yüce Dağ Başında,258315,2025-05-11,2025,5,11,Sunday,16,5,True
3801,2025-05-11 16:15:00,Jens Lekman,"How We Met, the Long Version",256266,2025-05-11,2025,5,11,Sunday,16,15,True
3802,2025-05-11 16:18:00,Robert Palmer,Johnny And Mary,239053,2025-05-11,2025,5,11,Sunday,16,18,True


In [106]:
lst = [1, 2, 3]

square = lambda x: x**2

list(map(square, lst))

[1, 4, 9]

In [ ]:
# pandas

df = pd.DataFrame({"a": [1, 2, 3]})

df["a"].map(square)

0    1
1    4
2    9
Name: a, dtype: int64